# Performance Optimization and Solver Tuning

This notebook demonstrates performance optimization techniques for ws3 optimization problems.

**What You'll Learn:**

- How to tune solver parameters for faster solves
- How to profile memory usage and detect leaks
- How to benchmark solver performance
- How to analyze parallel speedup
- How to use result caching for repeated scenarios
- How to warm-start from previous solutions

**Prerequisites:** Completion of `070_ws3_quickstart_complete_workflow.ipynb`

**Note:** This notebook requires the `ws3.perf` module which provides performance optimization utilities.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
import ws3.opt
from ws3.perf import (
    SolverTuner,
    MemoryProfiler,
    PerformanceBenchmark,
    ResultCache,
    IncrementalSolver,
    tune_solver,
    profile_memory,
    benchmark,
    cache_results,
    incremental_solve
)

print("Imports complete!")

## 1. Solver Parameter Tuning

Solver parameters significantly impact optimization performance. Different solvers (Gurobi, CBC, GLPK, HiGHS) have different parameter sets that can be tuned for specific problem types.

**Why Tune Solvers?**

- Forest optimization problems have characteristic structures (many binary variables, network-like constraints)
- Default parameters may not be optimal for your specific problem
- Tuning can reduce solve time by 50% or more
- Different problem sizes benefit from different parameter settings

**Key Parameters to Tune:**

- `Threads`: Number of CPU cores to use
- `TimeLimit`: Maximum solve time (seconds)
- `MIPGap`: Optimality gap tolerance (lower = more optimal but slower)
- `Presolve`: Aggressiveness of problem reduction before solving
- `Cuts`: Type and aggressiveness of cutting planes

In [ ]:
# Create a small forest model for performance testing
fm = ws3.forest.ForestModel(
    model_name="tsa24",
    model_path="data/woodstock_model_files_tsa24",
    base_year=2020,
    horizon=10,
    period_length=10,
    max_age=1000
)

fm.import_landscape_section()
fm.import_areas_section(convert_periods_to_years=10)
fm.import_yields_section(convert_periods_to_years=10)
fm.import_actions_section(convert_periods_to_years=10)
fm.import_transitions_section(convert_periods_to_years=10)
fm.initialize_areas()
fm.add_null_action()
fm.reset_actions()

# Compile a simple scenario
problem = ws3.opt.compile_scenario(
    fm,
    scenario_name="perf_test",
    objective="maximize_npv",
    weights={"npv": 1.0}
)

print(f"Problem created:")
print(f"  Variables: {len(problem._vars)}")
print(f"  Constraints: {len(problem._constraints)}")
print(f"  Solver: {problem._solver}")

## 2. Memory Profiling

Memory profiling helps identify:
- Memory leaks in long-running optimization workflows
- Peak memory usage for capacity planning
- Memory growth between solve operations

**Why Profile Memory?**

- Large forest models (1000+ DTs) can consume significant memory
- Repeated solves may accumulate memory if not cleaned up
- Understanding memory usage helps scale to larger problems
- Detects potential memory leaks early

**Key Metrics:**

- Current memory usage (MB)
- Peak memory usage (MB)
- Process memory (includes Python overhead)
- Memory delta between operations

In [ ]:
# Create memory profiler
profiler = profile_memory()

# Take initial snapshot
snapshot_before = profiler.take_snapshot('initial')
print(f"Initial memory: {snapshot_before['current_mb']:.2f} MB")

# Profile a solve operation
def solve_problem():
    problem.solve(threads=1)

profile_result = profiler.profile_solve(solve_problem)

# Take final snapshot
snapshot_after = profiler.take_snapshot('after_solve')
print(f"After solve: {snapshot_after['current_mb']:.2f} MB")
print(f"Memory delta: {profile_result['memory_delta']:.2f} MB")
print(f"Solve time: {profile_result['solve_time']:.2f}s")

# Show profiling report
report = profiler.get_report()
if not report.empty:
    print("\nMemory Profiling Report:")
    display(report)

## 3. Performance Benchmarking

Benchmarking provides standardized performance metrics for:
- Comparing solver configurations
- Tracking performance regressions
- Setting performance baselines
- Validating optimization improvements

**Key Metrics:**

- Solve time (mean, std, min, max)
- Solution quality (objective value, optimality gap)
- Solver status (optimal, infeasible, unbounded)
- Consistency across multiple runs

**When to Benchmark:**

- After changing solver parameters
- When updating solver versions
- When scaling to larger problems
- Before and after code optimizations

In [ ]:
# Create benchmark instance
bench = benchmark(problem)

# Run basic benchmark (5 runs)
print("Running benchmark (5 runs)...")
results = bench.benchmark_solve(n_runs=5, threads=1)

print(f"\nBenchmark Results:")
print(f"  Mean time: {results['mean_time']:.2f}s")
print(f"  Std dev: {results['std_time']:.2f}s")
print(f"  Min time: {results['min_time']:.2f}s")
print(f"  Max time: {results['max_time']:.2f}s")
print(f"  Status: {results['status']}")

# Display results as DataFrame
results_df = pd.DataFrame([results])
display(results_df)

## 4. Parallel Speedup Analysis

Parallel optimization can significantly reduce solve times for large problems. However, speedup is not always linear due to:

- Communication overhead between threads
- Load balancing issues
- Solver algorithm limitations
- Problem structure (some problems parallelize better than others)

**Expected Behavior:**

- Small problems: Minimal speedup (overhead dominates)
- Medium problems: Near-linear speedup up to ~8 cores
- Large problems: Sub-linear but significant speedup (2-4x with 8 cores)

**When Parallelism Helps Most:**

- Large MIP problems (10,000+ variables)
- Problems with many integer variables
- Problems with loose optimality gaps
- When time is more important than optimality

In [ ]:
# Benchmark parallel speedup
print("Testing parallel speedup with different thread counts...")
parallel_results = bench.benchmark_parallel(threads_list=[1, 2, 4, 8])

print("\nParallel Benchmark Results:")
display(parallel_results)

# Calculate and display speedup
speedups = bench.get_speedup(baseline_threads=1)
print("\nSpeedup Analysis:")
for threads, speedup in speedups.items():
    print(f"  {threads} threads: {speedup:.2f}x speedup")

# Plot speedup curve
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
bench.plot_speedup(ax=ax)
plt.tight_layout()
plt.show()

## 5. Result Caching

Result caching stores optimization solutions for repeated scenarios with identical parameters. This is useful for:

- **Scenario analysis**: When comparing many similar scenarios
- **Sensitivity analysis**: Testing parameter variations
- **Interactive workflows**: Quick re-exploration of known configurations
- **Batch processing**: Avoiding redundant solves

**How It Works:**

1. Compute a unique cache key from problem configuration
2. Check if result exists in cache
3. If yes: Return cached result (instant)
4. If no: Solve, store result, return solution

**Cache Key Components:**

- Number of variables
- Number of constraints
- Objective function parameters
- Constraint parameters

In [ ]:
# Create cache instance
cache = cache_results(cache_dir='.perf_test_cache')

# Clear any existing cache
cache.clear()
print(f"Cache cleared. Stats: {cache.stats()}")

# First solve (cache miss)
print("\nFirst solve (cache miss)...")
start = time.time()
problem.solve(threads=1)
time_first = time.time() - start
print(f"Time: {time_first:.2f}s")

# Store in cache
cache.put(problem, problem._solution)
print(f"Stored in cache. Stats: {cache.stats()}")

# Second solve (cache hit)
print("\nSecond solve (cache hit)...")
start = time.time()
cached_result = cache.get(problem)
time_cached = time.time() - start
print(f"Time: {time_cached:.4f}s")

print(f"\nSpeedup: {time_first / time_cached:.0f}x faster with cache!")

## 6. Warm Starting (Incremental Solving)

Warm starting provides an initial solution to the solver, which can dramatically reduce solve time when:

- Solving similar problems in sequence
- Performing sensitivity analysis
- Running scenario comparisons
- Re-optimizing after small parameter changes

**How Warm Starting Works:**

1. Solve initial problem to optimality
2. Extract solution values
3. Modify problem slightly (change objective, add constraint, etc.)
4. Provide previous solution as starting point
5. Solver uses this as initial feasible solution

**Benefits:**

- Can reduce solve time by 50-90% for similar problems
- Enables solving larger problems within time limits
- Useful for interactive optimization workflows
- Essential for some decomposition methods

**Limitations:**

- Only works with MIP solvers that support warm starts (Gurobi, CBC)
- Solution quality depends on how similar the new problem is
- Not all solver configurations support warm starts

In [ ]:
# Create incremental solver
inc_solver = incremental_solve(problem)

# Solve baseline problem
print("Solving baseline problem...")
start = time.time()
problem.solve(threads=1)
time_baseline = time.time() - start
print(f"Baseline time: {time_baseline:.2f}s")

# Extract solution for warm start
baseline_solution = problem._solution.copy()
print(f"Baseline objective: {problem._model.getAttr('ObjVal', problem._model) if hasattr(problem, '_model') else 'N/A'}")

# Warm start with same solution (simulating a similar problem)
print("\nWarm starting with baseline solution...")
inc_solver.warm_start(baseline_solution)

# Solve again (should be faster)
start = time.time()
problem.solve(warm_start=list(baseline_solution.values()), threads=1)
time_warm = time.time() - start
print(f"Warm start time: {time_warm:.2f}s")

print(f"\nSpeedup: {time_baseline / time_warm:.2f}x")

## Summary and Recommendations

**Key Takeaways:**

1. **Solver Tuning**: Always tune solver parameters for your specific problem type and size
2. **Memory Profiling**: Profile memory for large problems (1000+ DTs) to detect leaks
3. **Benchmarking**: Establish performance baselines and track regressions
4. **Parallel Speedup**: Expect 2-4x speedup with 8 cores for large problems
5. **Caching**: Use for repeated scenarios with identical parameters
6. **Warm Starting**: Use for sequential similar problems (50-90% speedup)

**When to Use Each Technique:**

| Technique | When to Use | Expected Benefit |
|-----------|-------------|------------------|
| Solver Tuning | First time solving a problem type | 20-50% faster |
| Memory Profiling | Large problems, long runs | Prevent crashes |
| Benchmarking | Comparing configurations | Quantify improvements |
| Parallel Solving | Large problems, time pressure | 2-4x speedup |
| Caching | Repeated identical solves | Instant results |
| Warm Starting | Similar sequential solves | 50-90% faster |

**Best Practices:**

- Start with solver tuning before trying other techniques
- Profile memory for problems with 1000+ DTs
- Benchmark before and after any optimization changes
- Use parallel solving for large problems (>5000 variables)
- Cache results for interactive scenario exploration
- Warm start when solving sequences of similar problems

In [ ]:
# Create summary table
summary_data = {
    'Technique': ['Solver Tuning', 'Memory Profiling', 'Benchmarking', 'Parallel Solving', 'Caching', 'Warm Starting'],
    'When to Use': [
        'First time solving',
        'Large problems',
        'Comparing configs',
        'Large problems, time pressure',
        'Repeated identical solves',
        'Similar sequential solves'
    ],
    'Expected Benefit': ['20-50% faster', 'Prevent crashes', 'Quantify improvements', '2-4x speedup', 'Instant results', '50-90% faster']
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print("\n" + "="*80)
print("Performance optimization notebook complete!")
print("="*80)